# NOTEBOOK FEATURE ENGINEERING

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import f_oneway

import os
import re

from IPython.display import display, Markdown

import missingno as msno
import sys

from rapidfuzz import process, fuzz

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from itertools import product

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r"..\00_Data\00_Processed\df_eda.csv")

In [3]:
df.columns

Index(['sex', 'race', 'age_cat', 'decile_score', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'score_text', 'screening_date', 'two_year_recid',
       'total_priors_count', 'c_charge_degree', 'start', 'end', 'event',
       'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'person_id',
       'agency_text', 'maritalstatus', 'language', 'rawscore', 'age',
       'juv_priors_count', 'adult_priors_count'],
      dtype='object')

In [4]:
lista_variables_modelo = [
    'person_id',
    'decile_score',
    'rawscore',
    'v_decile_score',
    'is_recid',
    'is_violent_recid',
    'two_year_recid',
    'sex',
    'race',
    'age',
    'total_priors_count',
    'adult_priors_count',
    'juv_priors_count',
    'c_charge_degree',
    'maritalstatus',
    'juv_fel_count',
    'juv_misd_count',
    'juv_other_count',
]

In [5]:
df = df[lista_variables_modelo]

In [6]:
df.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age',
       'total_priors_count', 'adult_priors_count', 'juv_priors_count',
       'c_charge_degree', 'maritalstatus', 'juv_fel_count', 'juv_misd_count',
       'juv_other_count'],
      dtype='object')

In [7]:
df['race'] = df['race'].replace({'asian': 'other', 'native american': 'other'})

In [8]:
df.race.value_counts()

race
african-american    3096
caucasian           2100
hispanic             561
other                380
Name: count, dtype: int64

In [9]:
df["two_year_recid"].sum()

np.int64(2219)

In [10]:
df.shape

(6137, 18)

In [11]:
df['sex'] = df['sex'].replace({'male': 0, 'female': 1})

In [12]:
df.sex.value_counts()

sex
0    4941
1    1196
Name: count, dtype: int64

In [13]:
df['c_charge_degree'] = df['c_charge_degree'].replace({'felony': 0, 'misdemeanor': 1})

In [14]:
df.c_charge_degree.value_counts()

c_charge_degree
0    3884
1    2253
Name: count, dtype: int64

In [15]:
df['maritalstatus'] = df['maritalstatus'].replace({'married': 'significant other', 'divorced': 'separated', 'widowed': 'other', 'unknown': 'other'})

In [16]:
df.maritalstatus.value_counts()

maritalstatus
single               4736
significant other     939
separated             408
other                  54
Name: count, dtype: int64

In [17]:
def one_hot_encoding(df, column, drop_val):
    encoder = OneHotEncoder(
    drop=[drop_val], 
    sparse_output=False
    )

    encoded = encoder.fit_transform(df[[column]])

    df_encoded = pd.DataFrame(
    encoded,
    columns = encoder.get_feature_names_out([column])
    )

    return df_encoded

In [18]:
df_model = pd.concat([df, one_hot_encoding(df, 'maritalstatus', 'single')], axis = 1)

In [19]:
df_model["log_juv_priors_count"] = np.log1p(df_model["juv_priors_count"])
df_model["log_adult_priors_count"] = np.log1p(df_model["adult_priors_count"])

In [20]:
#df_no_caucasian = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'caucasian')], axis = 1)

In [21]:
#df_model = pd.concat([df_marital_status, one_hot_encoding(df, 'race', 'african-american')], axis = 1)

In [22]:
df_model['priors_freq']=(df_model['total_priors_count']/(df_model['age']))


In [23]:
df_model["log_juv_priors_freq"] = np.where(
    df_model["age"] < 18,
    df_model["log_juv_priors_count"] / df_model["age"], # Si es menor
    df_model["log_juv_priors_count"] / 17              # Si es adulto
)


df_model["log_adult_priors_freq"] = np.where(
    df_model["age"] < 18,
    0,                                                
    df_model["log_adult_priors_count"] / (df_model["age"] - 17) 
)

In [24]:
df_model.priors_freq.max()

1.125

In [25]:
df_model['priors_freq_2']=((df_model['total_priors_count']+(df_model['total_priors_count'].mean()))/((df_model['age'])+(df_model['age']).mean()))

In [26]:
tasa_media = df_model["total_priors_count"].sum()/(df_model['age']).sum()

In [27]:
tasa_media

np.float64(0.09879730243541254)

In [28]:
beta = 5

In [29]:
alfa = tasa_media * beta

In [30]:
df_model['priors_freq_3']=((df_model['total_priors_count']+ alfa)/((df_model['age'])+ beta))


In [31]:
display(df_model.priors_freq_2.max())
display(df_model.priors_freq_3.max())

0.5865199992395725

0.9480685004198988

In [32]:
df_model.head()

,person_id,decile_score,rawscore,v_decile_score,is_recid,is_violent_recid,two_year_recid,sex,race,age,...,maritalstatus_other,maritalstatus_separated,maritalstatus_significant other,log_juv_priors_count,log_adult_priors_count,priors_freq,log_juv_priors_freq,log_adult_priors_freq,priors_freq_2,priors_freq_3
0,62384.0,2,-3.03,1,1,0,1,0,hispanic,94,...,1.0,0.0,0.0,0.0,1.098612,0.021277,0.0,0.014268,0.041225,0.025192
1,56279.0,1,-4.08,1,0,0,0,0,caucasian,80,...,0.0,1.0,0.0,0.0,1.609438,0.050000,0.0,0.025547,0.064119,0.052870
2,50959.0,1,-4.50,1,0,0,0,0,hispanic,79,...,0.0,0.0,1.0,0.0,0.000000,0.000000,0.0,0.000000,0.028842,0.005881
3,53038.0,1,-4.63,1,0,0,0,0,caucasian,77,...,0.0,0.0,1.0,0.0,0.000000,0.000000,0.0,0.000000,0.029368,0.006024
4,56006.0,1,-4.05,1,0,0,0,0,caucasian,76,...,0.0,1.0,0.0,0.0,0.693147,0.013158,0.0,0.011748,0.038849,0.018444


In [33]:
P_A = df_model["race"].value_counts(normalize=True)
P_Y = df_model["two_year_recid"].value_counts(normalize=True)
P_AY = df_model.groupby(["race","two_year_recid"]).size() / len(df_model)

#calcular pesos
def compute_weight(row):
    return (P_A[row["race"]] * P_Y[row["two_year_recid"]]) / P_AY[row["race"], row["two_year_recid"]]

df_model["weight"] = df_model.apply(compute_weight, axis=1)

In [34]:
df_model.columns

Index(['person_id', 'decile_score', 'rawscore', 'v_decile_score', 'is_recid',
       'is_violent_recid', 'two_year_recid', 'sex', 'race', 'age',
       'total_priors_count', 'adult_priors_count', 'juv_priors_count',
       'c_charge_degree', 'maritalstatus', 'juv_fel_count', 'juv_misd_count',
       'juv_other_count', 'maritalstatus_other', 'maritalstatus_separated',
       'maritalstatus_significant other', 'log_juv_priors_count',
       'log_adult_priors_count', 'priors_freq', 'log_juv_priors_freq',
       'log_adult_priors_freq', 'priors_freq_2', 'priors_freq_3', 'weight'],
      dtype='object')

In [35]:
df_model.to_csv(r'..\00_Data\00_Processed\df_modelo.csv', index=False)